# 01 — The second-order view: gradients and curvature

*Chapter 09 · XGBoost · Notebook 1 of 5*

Welcome to XGBoost. Before any new machinery, hold onto one fact: **XGBoost is not a new algorithm.**
It is the gradient-boosting engine you built in Chapter 08, sharpened in a few concrete ways and
engineered to run fast. This chapter earns each sharpening by hand, one notebook at a time.

The first sharpening is the subject of this notebook: use the loss's **curvature**, not only its
slope. In Chapter 08 you fit trees to the negative gradient (the slope). Once, for classification, you
also needed a *correction* — the Newton leaf — and we said the library applied it silently. By the end
of this notebook that correction will stop looking like a special case: it is one half of a single,
general rule, and that rule is the heart of XGBoost.

**Prerequisites.** Chapter 08 NB 2 (the residual *is* the negative gradient; boosting is gradient
descent in function space); Chapter 08 NB 3 (the Newton leaf-step for classification); Chapter 04
(regression trees, whose leaves hold the mean).

**What you'll be able to do.**
- Approximate any twice-differentiable loss to second order and read off the best step `w* = -g/h`.
- Extend that to a whole leaf: the optimal leaf weight is `w* = -G/H`.
- Show that Chapter 08's *two* leaf rules are this *one* rule, a single loss apart.
- Reproduce that leaf weight with XGBoost — once its regularizer is switched off.

## Where Chapter 08 left us

Gradient boosting builds a model in steps: start from a constant, then add shallow trees, each fit to
the **negative gradient** of the loss at the current prediction. Two facts from that chapter matter
here.

- **Regression (squared error).** The negative gradient was the residual `y - F`, and each tree's leaf
  held the **mean** of the residuals falling in it (Chapter 08 NB 1).
- **Classification (log-loss).** The negative gradient was the pseudo-residual `y - p`, but the mean
  was *not* the right leaf value — because log-loss **curves**, where squared error does not. NB 3
  needed a correction — the **Newton leaf** `Σ(y - p) / Σ p(1 - p)` — and noted the library applies it
  for you.

Two losses, two different-looking leaf rules. This notebook shows they are **one** rule. That rule is
what XGBoost is built on, so it is worth deriving with our own hands.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()

SEED = 0


def sigmoid(z):
    """Map a real-valued score to a probability in (0, 1)."""
    return 1.0 / (1.0 + np.exp(-z))


## The idea: approximate the loss to second order

We have a current prediction `F` for a point, and we want to nudge it by a step `w` to lower the loss
`L`. Instead of guessing, we look at the loss *near* `F`. Any smooth function is well described nearby
by its value, its slope, and its curvature — a **second-order Taylor approximation**:

`L(F + w) ≈ L(F) + g·w + ½ h·w²`

where `g = ∂L/∂F` is the **gradient** (the slope) and `h = ∂²L/∂F²` is the **Hessian** (the curvature).
As a function of the step `w`, the right-hand side is a parabola; as long as the curvature `h` is
positive — true for both losses in this notebook — it opens upward and has a single lowest point. We
minimise it exactly by setting the derivative to zero, `g + h·w = 0`, and solving:

`w* = -g / h`

The gradient tells us **which way** to step; the curvature tells us **how far**. A flat loss (small
`h`) lets us step boldly; a sharply curved loss (large `h`) calls for a cautious step. A pure gradient
step knows the direction but has to guess the distance — dividing by `h` is the guess made principled.
This is **Newton's method**, and we are about to see it is exactly the move Chapter 08 made for
classification. (The rule rests on `h > 0`: a loss with no curvature — absolute error, which you'll
meet in the exercises — has `h = 0`, and a pure second-order step has nothing to divide by.)

In [ ]:
# A single leaf of three points with mixed labels, all currently scored at F = 0
# (so p = 0.5 each). What single step w most lowers the leaf's log-loss?
y_leaf = np.array([1.0, 1.0, 0.0])
F_leaf = 0.0
p_leaf = sigmoid(np.full_like(y_leaf, F_leaf))  # all 0.5
g = p_leaf - y_leaf          # gradient of log-loss:  g = p - y
h = p_leaf * (1.0 - p_leaf)  # curvature:             h = p(1 - p)
G, H = g.sum(), h.sum()
w_newton = -G / H            # the one second-order step
w_true = np.log(2.0)         # the leaf's exact optimum: logit(2/3)


def leaf_loss(w):
    """Total log-loss of the leaf when a constant step w is added to every score.

    Accepts a scalar or a 1-D array of steps and returns a matching shape.
    """
    w_arr = np.atleast_1d(np.asarray(w, dtype=float))
    P = sigmoid(F_leaf + w_arr)
    per_point = -(y_leaf[:, None] * np.log(P) + (1 - y_leaf[:, None]) * np.log(1 - P))
    total = per_point.sum(axis=0)
    return total if np.ndim(w) else total[0]


ws = np.linspace(-1.5, 2.5, 400)
L0 = float(leaf_loss(0.0))
quad = L0 + G * ws + 0.5 * H * ws**2

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(ws, leaf_loss(ws), color=COLORS["model"], linewidth=2.6, label="true leaf loss  L(F + w)")
ax.plot(ws, quad, color=COLORS["highlight"], linewidth=2.0, linestyle="--",
        label="second-order approximation")
ax.axvline(w_newton, color=COLORS["class_b"], linewidth=1.8, label=f"w* = -G/H = {w_newton:.3f}")
ax.axvline(w_true, color=COLORS["muted"], linewidth=1.6, linestyle=":",
           label=f"true optimum = {w_true:.3f}")
ax.set_xlabel("step  w  added to every score in the leaf")
ax.set_ylabel("leaf log-loss")
ax.legend()
plt.show()

print(f"G = Σg = {G:.3f}   H = Σh = {H:.3f}")
print(f"one second-order step  w* = -G/H = {w_newton:.4f}")
print(f"the leaf's exact optimum         = {w_true:.4f}")


**Read the figure.** Near `w = 0` the dashed parabola and the true loss are nearly
indistinguishable — the second-order approximation is faithful where it matters. Its minimum,
`w* = -G/H ≈ 0.667`, sits very close to the leaf's exact optimum `≈ 0.693`. Close, but not identical:
one second-order step is the best move *for the local parabola*, not the final answer. That gap is
exactly why boosting takes **many** small steps rather than one — each tree improves the approximation
the next one starts from. The curvature `h` set the step's length: here `h = 0.25` (small), so the step
was generous.

In [ ]:
# Two ways to set the leaf's step from the SAME gradients
# (g, h, G, H, leaf_loss and ws come from the cell above).
w_blind = float((-g).mean())   # mean of the negative gradient: the step if curvature were 1
w_newton = float(-G / H)       # the second-order step: divide by the true curvature H
w_true = float(np.log(2.0))    # the leaf's exact optimum

loss_blind = float(leaf_loss(w_blind))
loss_newton = float(leaf_loss(w_newton))
loss_true = float(leaf_loss(w_true))

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(ws, leaf_loss(ws), color=COLORS["model"], linewidth=2.6, label="true leaf loss")
ax.plot(w_blind, loss_blind, "o", color=COLORS["error"], markersize=12, zorder=5,
        label=f"curvature-blind step (mean residual) = {w_blind:.3f}")
ax.plot(w_newton, loss_newton, "o", color=COLORS["class_b"], markersize=12, zorder=5,
        label=f"second-order step  -G/H = {w_newton:.3f}")
ax.axvline(w_true, color=COLORS["muted"], linewidth=1.6, linestyle=":",
           label=f"true optimum = {w_true:.3f}")
ax.set_xlabel("step  w  added to every score in the leaf")
ax.set_ylabel("leaf log-loss")
ax.legend()
plt.show()

print(f"curvature-blind (mean residual) step = {w_blind:.4f}  ->  loss {loss_blind:.4f}")
print(f"second-order (Newton) step           = {w_newton:.4f}  ->  loss {loss_newton:.4f}")
print(f"true leaf optimum                    = {w_true:.4f}  ->  loss {loss_true:.4f}")


**Read the figure.** The red point is the step you would take by treating the leaf like a
squared-error problem: the plain mean of the residuals. (Here the negative gradient `-g = y - p` *is*
the pseudo-residual, so "the mean of the negative gradient" and "the mean residual" name the same
number.) For log-loss it lands well short of the minimum, because it implicitly assumes a curvature of
`1` when the true curvature here is only `0.25`. Dividing the gradient sum by the *real* curvature `H`
rescales the step (by a factor of `1 / 0.25 = 4`) onto the bottom of the curve. Same direction,
corrected distance. This rescaling is the single move Chapter 08 NB 3 made for classification — and we
are about to see both chapters' leaf rules fall out of it.

## From one point to a whole leaf

A leaf does not hold one point; it holds many, and they all receive the *same* leaf weight `w`. So we
add up the per-point approximations:

`Σᵢ [ gᵢ·w + ½ hᵢ·w² ] = (Σᵢ gᵢ)·w + ½ (Σᵢ hᵢ)·w²`

Writing `G = Σᵢ gᵢ` and `H = Σᵢ hᵢ` for the leaf's totals, this is the same parabola as before, and its
minimiser is the same formula:

`w* = -G / H`

One rule for the best constant on a leaf, valid for **any** twice-differentiable loss. The loss enters
only through the two sums `G` and `H`. Let us feed it the two losses from Chapter 08.

## Pinning the convention

We will be explicit about signs once, so every later line is unambiguous:

- gradient `gᵢ = ∂L/∂Fᵢ`,
- Hessian `hᵢ = ∂²L/∂Fᵢ²`,
- leaf weight `w* = -G/H`.

The minus sign is what turns Chapter 08's *negative* gradient into the *positive* update we actually
add. With that fixed, the loss supplies `(g, h)` and the rule does the rest.

## Loss 1 — squared error

For regression with `L = ½ (y - F)²`:

- `g = ∂L/∂F = F - y` (the negative residual),
- `h = ∂²L/∂F² = 1`.

The curvature is `1` everywhere. So the rule gives

`w* = -G/H = -Σ(F - y) / n = Σ(y - F) / n` = the **mean residual**.

For squared error the second-order step is exactly the mean. That is why a regression tree — whose
leaves already hold the mean — needed no correction in Chapter 08 NB 1, and why curvature was never
mentioned there. Let us confirm it on that very notebook's data.

In [ ]:
# Chapter 08 NB 1's regression problem: a sine through noise.
rng = np.random.default_rng(SEED)
n = 120
x = np.sort(rng.uniform(0.0, 2.0 * np.pi, n))
X = x.reshape(-1, 1)
y = np.sin(x) + rng.normal(0.0, 0.25, n)

F0 = y.mean()                       # the constant starting model
residual = y - F0
g_reg = F0 - y                      # gradient of 1/2 (y - F)^2 at F = F0
h_reg = np.ones(n)                  # curvature is 1 everywhere

# The first boosting tree fits the residuals; read off its leaves.
tree = DecisionTreeRegressor(max_depth=2, random_state=SEED).fit(X, residual)
leaves = tree.apply(X)
leaf_pred = tree.predict(X)

print(f"F0 = mean(y) = {F0:.4f}\n")
print(f"{'leaf':>5} {'n':>4} {'-G/H':>11} {'mean residual':>15} {'tree leaf value':>17}")
for lid in np.unique(leaves):
    m = leaves == lid
    w_second_order = -g_reg[m].sum() / h_reg[m].sum()
    print(f"{lid:>5} {m.sum():>4} {w_second_order:>11.5f} {residual[m].mean():>15.5f} "
          f"{leaf_pred[m][0]:>17.5f}")


**Read the table.** Three columns, one number per leaf. The second-order weight `-G/H`, the mean
of the residuals, and the value the regression tree actually stored are identical to five decimals. For
squared error the tree's native leaf — the mean — already *is* the second-order optimum, because the
curvature is a flat `1`. No correction was ever needed, which is why Chapter 08 NB 1 could stay silent
about curvature.

## Loss 2 — log-loss

For binary classification with log-loss, working in the score (log-odds) `F` with `p = sigmoid(F)`:

- `g = ∂L/∂F = p - y` (the negative pseudo-residual),
- `h = ∂²L/∂F² = p(1 - p)`.

Now the curvature **varies** from point to point. A confident point (`p` near `0` or `1`) has tiny
curvature; an uncertain one (`p ≈ 0.5`) has the largest, `0.25`. The rule gives

`w* = -G/H = Σ(y - p) / Σ p(1 - p)`

which is exactly the **Newton leaf** of Chapter 08 NB 3. The mean residual would assume `h = 1` and
fall short, as we saw; dividing by the real `Σ p(1 - p)` is the correction. Let us confirm on that
notebook's data.

In [ ]:
# Chapter 08 NB 3's classification problem: the two moons, same split.
X_moons, y_moons = make_moons(n_samples=400, noise=0.20, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_moons, y_moons, test_size=0.30, random_state=SEED, stratify=y_moons
)

p0 = y_tr.mean()
F0_clf = np.log(p0 / (1 - p0))   # starting score = log-odds of the base rate
p = sigmoid(np.full(len(y_tr), F0_clf))
pseudo_residual = y_tr - p       # negative gradient
g_clf = p - y_tr                 # gradient of log-loss
h_clf = p * (1 - p)              # curvature

tree_clf = DecisionTreeRegressor(max_depth=2, random_state=SEED).fit(X_tr, pseudo_residual)
leaves_clf = tree_clf.apply(X_tr)

print(f"p0 = mean(y_train) = {p0:.3f}   ->   F0 = log-odds = {F0_clf:.4f}\n")
print(f"{'leaf':>5} {'n':>4} {'-G/H':>11} {'Newton  Σ(y-p)/Σp(1-p)':>26}")
for lid in np.unique(leaves_clf):
    m = leaves_clf == lid
    w_second_order = -g_clf[m].sum() / h_clf[m].sum()
    newton = pseudo_residual[m].sum() / (p[m] * (1 - p[m])).sum()
    print(f"{lid:>5} {m.sum():>4} {w_second_order:>11.5f} {newton:>26.5f}")


In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))

# Left: the curvature h as a function of the current score F.
F_grid = np.linspace(-6, 6, 400)
p_grid = sigmoid(F_grid)
axL.plot(F_grid, np.ones_like(F_grid), color=COLORS["class_a"], linewidth=2.6,
         label="squared error:  h = 1")
axL.plot(F_grid, p_grid * (1 - p_grid), color=COLORS["class_b"], linewidth=2.6,
         label="log-loss:  h = p(1 - p)")
axL.set_xlabel("current score  F")
axL.set_ylabel("curvature  h")
axL.set_title("how sharply each point curves the loss")
axL.legend()

# Right: the per-point weight that curvature gives in the leaf average.
F_pts = np.array([-3.0, -1.0, 0.0, 1.0, 3.0])
p_pts = sigmoid(F_pts)
h_pts = p_pts * (1 - p_pts)
axR.bar(range(len(F_pts)), h_pts, color=COLORS["class_b"], label="log-loss:  h = p(1 - p)")
axR.axhline(1.0, color=COLORS["class_a"], linewidth=2.2, linestyle="--",
            label="squared error:  h = 1")
axR.set_xticks(range(len(F_pts)))
axR.set_xticklabels([f"p = {pp:.2f}" for pp in p_pts])
axR.set_ylabel("curvature weight in the leaf")
axR.set_title("log-loss leans on the uncertain points")
axR.legend()
plt.show()


**Read the figure.** *Left:* squared error has a flat curvature of `1` — every point bends the
loss the same amount, at every score. Log-loss has the bell `p(1 - p)`: a point bends the loss most
when the model is unsure (`p ≈ 0.5`) and barely at all once the model is confident. *Right:* in the
leaf weight `-G/H`, that curvature is the weight each point carries. Squared error weights all points
equally, so `-G/H` is the plain mean. Log-loss weights the uncertain points most, so `-G/H` becomes the
Newton leaf.

This is the unification. Chapter 08's two leaf rules were never two rules — both are `w* = -G/H`. The
loss only changes the two ingredients `G` and `H`. That single second-order rule is what XGBoost runs
for **any** twice-differentiable loss you hand it.

In [ ]:
# A tiny regression toy, so we can read XGBoost's one tree by eye.
x_toy = np.array([[0.0], [1.0], [2.0], [3.0], [4.0], [5.0]])
y_toy = np.array([1.0, 1.2, 0.9, 5.0, 5.3, 4.8])
base = float(y_toy.mean())

# One tree, one split, learning_rate = 1, and the regularizer switched OFF (reg_lambda = gamma = 0).
reg = XGBRegressor(
    n_estimators=1, max_depth=1, learning_rate=1.0, reg_lambda=0.0, gamma=0.0,
    base_score=base, objective="reg:squarederror", tree_method="exact", random_state=SEED,
).fit(x_toy, y_toy)

dump = reg.get_booster().trees_to_dataframe()
split = float(dump.loc[dump["Feature"] != "Leaf", "Split"].iloc[0])

# By hand, on XGBoost's own split, with squared error (g = F - y, h = 1).
left = x_toy.ravel() < split
g_toy = base - y_toy
h_toy = np.ones_like(y_toy)
w_left = -g_toy[left].sum() / h_toy[left].sum()
w_right = -g_toy[~left].sum() / h_toy[~left].sum()
xgb_leaves = sorted(float(v) for v in dump.loc[dump["Feature"] == "Leaf", "Gain"])

print(f"base_score = mean(y) = {base:.3f};   XGBoost splits at  x < {split}")
print(f"by-hand  -G/H:    left = {w_left:+.5f}    right = {w_right:+.5f}")
print(f"XGBoost leaves:   {[round(v, 5) for v in xgb_leaves]}")


**Read it.** With its regularizer switched off, XGBoost's leaf weights are exactly our `-G/H`.
The library is running the same second-order rule we derived by hand. (In the next notebook we turn the
regularizer back on — `reg_lambda` — and watch the leaf weight shrink; that `+ λ` in the denominator is
the "regularized" in XGBoost.)

One detail we leaned on: we pinned `base_score` so the starting score `F₀` was a known constant, which
let us check the arithmetic. Left free, XGBoost *learns* `base_score` from the data — the same role
Chapter 08's `init_` played. We fixed it here only to make the hand-check clean.

## Your turn

1. **(warm-up)** A leaf holds five points with gradients `g = [0.2, -0.4, 0.1, -0.3, 0.5]` and Hessians
   `h = [1, 1, 1, 1, 1]`. Compute `G`, `H`, and the leaf weight `w* = -G/H` by hand. What loss has
   `h = 1`?
2. **(core)** Show algebraically that for squared error `-G/H` equals the mean residual for a leaf of
   *any* size. Then work out `(g, h)` for the absolute-error loss `L = |y - F|`, and say what its
   second derivative does to the rule `w* = -G/H`. (This is why absolute error is awkward for a pure
   second-order method.)
3. **(reach)** Using Figure 1, explain in your own words why a single second-order step does not land on
   the leaf's exact optimum for log-loss, and why a sequence of *shrunken* steps (the learning rate `ν`
   from Chapter 08 NB 4) does reach it. What role does the learning rate play in keeping each parabola a
   good approximation?

## What you built

You replaced Chapter 08's two leaf recipes with **one** second-order rule:

`w* = -G/H`   —   gradient for the direction, curvature for the step length.

- Squared error has `h = 1`, so `-G/H` is the plain **mean residual** (Chapter 08 NB 1).
- Log-loss has `h = p(1 - p)`, so `-G/H` is the **Newton leaf** (Chapter 08 NB 3).
- XGBoost, with its regularizer switched off, lands on exactly that weight.

The loss enters only through the sums `G` and `H`, so this one rule serves any twice-differentiable
loss — which is precisely the generality XGBoost is built to exploit.

**Vocabulary.** gradient `g = ∂L/∂F` · Hessian (curvature) `h = ∂²L/∂F²` · second-order / Newton step ·
leaf weight `-G/H`.

Next, we turn the regularizer **on**: the penalty `λ` shrinks the leaf weight to `-G/(H + λ)`, and a
second penalty `γ` decides whether a split is worth making at all.

## References

- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16. DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785). (Eq. 5–6: the second-order
  objective and the optimal leaf weight.)
- Friedman, J. H. (2001). *Greedy function approximation: a gradient boosting machine.* Annals of
  Statistics, 29(5), 1189–1232. DOI [10.1214/aos/1013203451](https://doi.org/10.1214/aos/1013203451).
  (TreeBoost: the Newton leaf-step as a one-step Newton–Raphson update.)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.10. DOI [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

*Previous: Chapter 08 — Gradient Boosting (the California-housing capstone).*  ·  *Next: 02 — The
regularized objective.*